# 07 · 张量自动求导（三）：反向引擎、no_grad 与端到端训练

> **本节属于 Part 3 · 张量自动求导。这是 Part 3 的收尾，也是 minitorch 引擎的"竣工验收"。**

前两节我们实现了 `Tensor` 的各种算子。本节把**反向引擎本身**讲透：

- 为什么 `backward()` 要先做**拓扑排序**？
- **梯度累加**（一个张量被多次使用）是怎么回事？
- `no_grad`（推理时不建图）与 `detach`（截断梯度）有什么用？

最后，我们用这套**完整引擎**端到端训练一个网络——为 Part 4 把训练样板封装成 `nn.Module` 做铺垫。

## 学习目标

- 理解反向引擎的两大机制：**拓扑排序** + **梯度累加（`+=`）**
- 掌握 `no_grad()` 与 `detach()` 的用途与区别
- 用完整的 `minitorch.Tensor` **端到端训练**一个二分类 MLP，并可视化决策边界

In [ ]:
import inspect
import numpy as np
import matplotlib.pyplot as plt
from minitorch import Tensor, no_grad, set_seed, rel_error

## 1. 反向引擎：拓扑排序

反向传播必须**保证顺序**：一个节点的梯度，只有在它**所有的下游**都算完后才完整。否则把"半成品"梯度往上游传就错了。

**拓扑排序**正好给出这样的顺序——把图排成"输入在前、输出在后"的线性序，反向时**逆序**遍历即可。下面是包里 `backward()` 的真实实现：

In [ ]:
print(inspect.getsource(Tensor.backward))

## 2. 梯度累加：为什么是 `+=`

如果一个张量被用在**多个**地方，它会从多条路径收到梯度，必须**累加**。这就是每个 `_backward` 里都用 `self.grad += ...` 的原因。

In [ ]:
x = Tensor(np.array([2.0, 3.0]))
# L = sum(x*x) + sum(x)  ->  dL/dx = 2x + 1
L = (x * x).sum() + x.sum()
L.backward()
print("x.grad =", x.grad, " 期望 2x+1 =", 2 * x.data + 1)
print("一致 ->", rel_error(x.grad, 2 * x.data + 1) < 1e-9)

> ⚠️ 正因为梯度会累加，所以训练时每一步都要先**清零梯度**（`p.grad = 0`），否则上一步的梯度会污染这一步。Part 4 的优化器会帮我们自动做这件事（`optimizer.zero_grad()`）。

## 3. no_grad：推理时不建图

训练时我们需要计算图来反向求梯度；但**推理/评估**时不需要梯度，建图纯属浪费内存和时间。`no_grad()` 上下文让其中的运算**不记录计算图**。

In [ ]:
x = Tensor(np.random.randn(4))
with no_grad():
    y = x * 2 + 1
print("no_grad 内：y 没有前驱节点 ->", len(y._prev) == 0)

# 对比：正常情况下会建图
z = x * 2 + 1
print("正常情况：z 有前驱节点   ->", len(z._prev) > 0)

## 4. detach：截断梯度

`detach()` 返回一个数值相同、但**脱离计算图**的新张量。常用于"我想用这个值，但不希望梯度从这里继续往回传"的场景（比如目标值、某些正则项）。

In [ ]:
a = Tensor(np.array([1.0, 2.0, 3.0]))
b = (a * 2).detach()      # b 与 a 的计算图断开
(b * 3).sum().backward()
print("a.grad =", a.grad, " （被 detach 截断，所以是 0）")

## 5. 竣工验收：端到端训练一个 MLP

现在用**完整的 minitorch 引擎**，从零训练一个两层 MLP，分类经典的 `make_moons`。注意：除了 `Tensor`，我们没用任何深度学习库——前向我们写，梯度引擎自动求，参数我们更新。

In [ ]:
def make_moons(n=200, noise=0.15, seed=0):
    rng = np.random.RandomState(seed)
    n0, n1 = n // 2, n - n // 2
    t0, t1 = np.linspace(0, np.pi, n0), np.linspace(0, np.pi, n1)
    outer = np.c_[np.cos(t0), np.sin(t0)]
    inner = np.c_[1 - np.cos(t1), 1 - np.sin(t1) - 0.5]
    X = np.vstack([outer, inner]) + rng.randn(n, 2) * noise
    y = np.array([-1.0] * n0 + [1.0] * n1).reshape(-1, 1)   # 标签 {-1,+1}
    return X, y

X, y = make_moons()
set_seed(0)
# 参数：2 -> 16 -> 1
W1 = Tensor(np.random.randn(2, 16) * 0.5); b1 = Tensor(np.zeros(16))
W2 = Tensor(np.random.randn(16, 1) * 0.5); b2 = Tensor(np.zeros(1))
params = [W1, b1, W2, b2]

def forward(Xt):
    H = (Xt @ W1 + b1).relu()
    return H @ W2 + b2          # 原始分数 (N,1)

history = []
for epoch in range(200):
    for p in params:
        p.grad = np.zeros_like(p.data)         # 梯度清零
    scores = forward(Tensor(X))
    # max-margin（hinge）损失 + L2 正则
    loss = (1 - Tensor(y) * scores).relu().mean() + (W1 * W1).sum() * 1e-3 + (W2 * W2).sum() * 1e-3
    loss.backward()
    for p in params:
        p.data -= 0.1 * p.grad                  # SGD 更新
    history.append(float(loss.data))

acc = ((forward(Tensor(X)).data > 0) == (y > 0)).mean()
print(f"最终 loss={history[-1]:.3f}, 训练准确率={acc*100:.1f}%")

In [ ]:
# 决策边界（推理用 no_grad，省事又省内存）
xx, yy = np.meshgrid(np.linspace(-1.5, 2.5, 200), np.linspace(-1.0, 1.5, 200))
with no_grad():
    Z = forward(Tensor(np.c_[xx.ravel(), yy.ravel()])).data.reshape(xx.shape)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(history); ax[0].set_yscale("log"); ax[0].set_title("Loss"); ax[0].grid(alpha=0.3)
ax[1].contourf(xx, yy, Z, levels=[-100, 0, 100], cmap="bwr", alpha=0.25)
ax[1].scatter(X[:, 0], X[:, 1], c=y.ravel(), cmap="bwr", s=16, edgecolors="k", linewidths=0.3)
ax[1].set_title(f"Decision boundary (acc={acc*100:.0f}%)")
plt.tight_layout(); plt.show()

## 📦 沉淀进 minitorch —— 引擎竣工

至此，`minitorch.Tensor` 已是一个**功能完整、经过全面梯度检查的张量自动求导引擎**：逐元素运算、广播、matmul、归约、激活、形状变换，加上 `backward / no_grad / detach`。

它的正确性由 `tests/test_tensor.py` 守护（每个算子都做数值梯度检查、并与 PyTorch 对照）。**这是整个 minitorch 框架的基石**——接下来的 nn 层、优化器、CNN、RNN、Transformer 全都建立在它之上。

In [ ]:
# 引擎的完整"武器库"
print("Tensor 公开方法:")
print(" ", [m for m in dir(Tensor) if not m.startswith("_")])

## 小练习

1. **加深加宽**：把隐藏层从 16 改成 `[32, 16]` 两层，决策边界更贴合月牙了吗？
2. **看 no_grad 的收益**：在训练循环外，分别用 `with no_grad()` 和不用，对全部数据各前向 1000 次，比较耗时（no_grad 不建图，应更快）。
3. **detach 的用处**：把损失里的正则项改成 `(W1.detach() * W1).sum()`，想想这会让梯度发生什么变化（提示：相当于把其中一个因子当常数）。

## 小结 & 下一站

✅ 我们讲透了反向引擎（拓扑排序 + 梯度累加），掌握了 `no_grad` 与 `detach`，并用**完整的 minitorch 引擎端到端训练**了一个 MLP——**Part 3 的"造心脏"任务圆满完成！**

但你也许注意到：训练循环里那些"梯度清零、参数更新、收集参数"的样板代码有点啰嗦。

**下一站 → Part 4 `08_module_and_linear`**：我们仿照 PyTorch，把这些样板封装成优雅的 `nn.Module / Linear / Sequential`——从此搭网络像搭积木一样简单。